In [ ]:
# %% [markdown]
# # 02 – Baseline RAG Accuracy Benchmark (can choose data_source)

# %%
from memorypoison_audit.source.core.agent_orchestrator import AgentOrchestrator
from memorypoison_audit.source.benchmarks.metrics import MetricsCalculator
from memorypoison_audit.source.data_loader import HotpotQALoader, SyntheticDataGenerator
from memorypoison_audit.experiments.notebooks.shared_functions import save_metrics, get_results_path
import memorypoison_audit.source.disable_warnings
import json, numpy as np

DATA_SOURCE = "synthetic"  # Change to "hotpot" for real data

session_id = "baseline"
agent = AgentOrchestrator(session_id, {})

if DATA_SOURCE == "hotpot":
    loader = HotpotQALoader()
    data = loader.load_dev()
    qa_pairs = data[:50]
    for item in qa_pairs:
        context = " ".join(item.get("context", []))
        if context:
            agent.memory_store.add_fact(session_id, context)
    questions = [item["question"] for item in qa_pairs]
    ground_truth = [item["answer"] for item in qa_pairs]
else:
    gen = SyntheticDataGenerator()
    facts = gen.generate_facts(num_facts=100)
    for fact in facts:
        agent.memory_store.add_fact(session_id, fact)
    qa_pairs = gen.generate_qa_pairs(num_pairs=50)
    questions = [item["question"] for item in qa_pairs]
    ground_truth = [item["answer"] for item in qa_pairs]

# %%
f1_scores = []
for q, a in zip(questions, ground_truth):
    retrieved = agent.retrieve_context(q)
    pred = retrieved[0] if retrieved else ""
    f1 = MetricsCalculator.f1_score_lists(pred, a)
    f1_scores.append(f1)
avg_f1 = np.mean(f1_scores)
print(f"Average F1: {avg_f1:.4f}")

# %%
save_metrics("baseline", {"avg_f1": avg_f1, "data_source": DATA_SOURCE}, data_source=DATA_SOURCE)